In [2]:
import pandas as pd

df = pd.read_csv("Airlines.csv.zip")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Airlines.csv.zip'

In [1]:
df.isnull().sum()

# Drop columns with too many missing values
df = df.dropna(axis=1, thresh=0.7*len(df))

# Fill remaining missing values
df = df.fillna(method='ffill')

NameError: name 'df' is not defined

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['Airline'] = le.fit_transform(df['Airline'])
df['Origin'] = le.fit_transform(df['Origin'])
df['Dest'] = le.fit_transform(df['Dest'])

In [ ]:
df = pd.get_dummies(df, columns=['Airline', 'Origin', 'Dest'])

In [ ]:
df['FlightDate'] = pd.to_datetime(df['FlightDate'])

df['Weekday'] = df['FlightDate'].dt.weekday
df['Month'] = df['FlightDate'].dt.month

df = df.drop(columns=['FlightDate'])

In [ ]:
df['Delay'] = df['ArrDelay'].apply(lambda x: 1 if x > 0 else 0)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Delay'])
y = df['Delay']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

print("Logistic Regression:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))

print("\nRandom Forest:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

y_prob = rf.predict_proba(X_test)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label="AUC="+str(roc_auc))
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_rf))

In [ ]:
import joblib

joblib.dump(rf, "flight_delay_model.pkl")